In [ ]:
#UPSC essay evaluation workflow
#START -> Clarity of thought (OT)
#START -> Depth of Analysis (DOA)
#START -> Language
#inputs -> Final eval -> END
#Send each of above to 3 LLMs

#Output
#Text feedback str for each input (Summarized)
#Score (0-10) int for each input (Averaged)

#Each input will be sent to one final eval node
#Merge  feedback for 3 input and generate summarized feedback
#and average score for the essay as final score

#Structured output
#Reducer function


In [23]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_anthropic import ChatAnthropic
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator

In [5]:
load_dotenv()
model = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    temperature=0
)

In [ ]:
#create a schema for structured output
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)

In [7]:
structured_model = model.with_structured_output(EvaluationSchema)

In [8]:
essay = """Artificial Intelligence (AI) is playing a major role in the growth and development of India. It is transforming many sectors such as healthcare, education, agriculture, transportation, and business. AI-powered technologies help doctors detect diseases early, improve medical research, and provide better healthcare services in rural areas. In education, AI-based learning platforms offer personalized learning experiences and help students access quality education online.

In agriculture, AI helps farmers predict weather conditions, monitor crop health, and increase productivity through smart farming techniques. AI is also improving transportation through traffic management systems, self-driving technology research, and better public transport planning. Indian businesses and startups are using AI to improve customer service, automate repetitive tasks, and increase efficiency.

The Indian government is also encouraging AI development through initiatives like Digital India and the National AI Strategy. AI is creating new job opportunities in fields such as data science, machine learning, robotics, and software development. However, AI also brings challenges such as job displacement, data privacy concerns, and the need for skilled professionals.

Overall, AI has the potential to make India more innovative, efficient, and globally competitive in the future.
"""

In [ ]:
prompt = f"Evaluate the lnaguage quality of the following essay and provide detailed feedback and a score out of 10:\n\n{essay}"
#structured_model.invoke(prompt)
#structured_model.invoke(prompt).score
#structured_model.invoke(prompt).feedback


'This essay demonstrates solid writing with clear organization and good coverage of AI\'s impact on India. Here are the strengths and areas for improvement:\n\nSTRENGTHS:\n- Clear structure with well-organized paragraphs that flow logically\n- Comprehensive coverage of multiple sectors (healthcare, education, agriculture, transportation, business)\n- Balanced perspective that acknowledges both benefits and challenges\n- Appropriate use of specific examples (Digital India, National AI Strategy)\n- Generally clear and accessible language for the target audience\n\nAREAS FOR IMPROVEMENT:\n\n1. VOCABULARY & EXPRESSION:\n- Some phrases are repetitive or generic ("playing a major role," "help/helps" appears 5 times)\n- Could use more varied and sophisticated vocabulary (e.g., "facilitating" instead of "help," "catalyzing" instead of "playing a role")\n- Phrases like "has the potential to make" are somewhat vague\n\n2. SENTENCE VARIETY:\n- Many sentences follow similar structures, particularl

In [14]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    avg_score: float
    

In [18]:
def evaluate_language(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the language quality of the following essay and provide detailed feedback and a score out of 10:\n\n{state['essay']}"
    output = structured_model.invoke(prompt)
    
    return  {'language_feedback': output.feedback, 'individual_scores': [output.score]}
    

In [19]:
def evaluate_analysis(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the depth of analysis of the following essay and provide detailed feedback and a score out of 10:\n\n{state['essay']}"
    output = structured_model.invoke(prompt)
    
    return  {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [20]:
def evaluate_thought(state: UPSCState) -> UPSCState:
    prompt = f"Evaluate the clarity of thought of the following essay and provide detailed feedback and a score out of 10:\n\n{state['essay']}"
    output = structured_model.invoke(prompt)
    
    return  {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [21]:
def final_evaluation(state: UPSCState) -> UPSCState:
    #Summary feedback
    prompt = f"Based on the following feedback and scores, provide a final evaluation of the essay:\n\nLanguage Feedback: {state['language_feedback']}\nAnalysis   Feedback: {state['analysis_feedback']}\nClarity Feedback: {state['clarity_feedback']}\nIndividual Scores: {state['individual_scores']}"
    overall_feedback = output = model.invoke(prompt).content
    
    #Average score
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])
    
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [25]:
graph = StateGraph(UPSCState)

#add node
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

#add edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')
graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('final_evaluation', END)

#compile Graph

workflow = graph.compile()








In [26]:
initial_state = {
    'essay': essay
}

final_state = workflow.invoke(initial_state)

print(final_state)
    

{'essay': 'Artificial Intelligence (AI) is playing a major role in the growth and development of India. It is transforming many sectors such as healthcare, education, agriculture, transportation, and business. AI-powered technologies help doctors detect diseases early, improve medical research, and provide better healthcare services in rural areas. In education, AI-based learning platforms offer personalized learning experiences and help students access quality education online.\n\nIn agriculture, AI helps farmers predict weather conditions, monitor crop health, and increase productivity through smart farming techniques. AI is also improving transportation through traffic management systems, self-driving technology research, and better public transport planning. Indian businesses and startups are using AI to improve customer service, automate repetitive tasks, and increase efficiency.\n\nThe Indian government is also encouraging AI development through initiatives like Digital India and